In [1]:
import diffrax
import jax
import jax.numpy as jnp
import jax.random as jrandom
import scipy.stats as stats
from diffrax import VirtualBrownianTree

In [2]:
key = jrandom.PRNGKey(5678)
# Get >80 randomly selected points; not too close to avoid discretisation error.
t0 = 0.3
t1 = 18.7
tree = VirtualBrownianTree(t0=t0, t1=t1, shape=(5,), tol=2**-7, key=key)
bm = tree.evaluate(1.5006555, 4.342863468, use_hh=True)
print(bm)
print(f"W: {bm.W}, H: {bm.H}")

BMInc(h=f32[], W=f32[5], J=None, H=f32[5])
W: [-0.01641206  2.8310525   2.6281621  -1.7422743  -1.0400537 ], H: [ 0.87908596 -0.63446015  0.39062464 -0.3203306   0.5385862 ]


In [3]:
bm_key, sample_key, permute_key = jrandom.split(key, 3)
tree = VirtualBrownianTree(t0=t0, t1=t1, shape=(), tol=2**-7, key=bm_key)
ts = jrandom.uniform(sample_key, shape=(200,), minval=t0, maxval=t1)
sorted_ts = jnp.sort(ts)
ts = []
prev_ti = sorted_ts[0]
for ti in sorted_ts[1:]:
    if ti < prev_ti + 2**-8:
        continue
    prev_ti = ti
    ts.append(ti)
ts = jnp.stack(ts)
assert len(ts) > 10

In [5]:
vals = jax.vmap(lambda ti, tj: tree.evaluate(ti, tj, use_hh=True))(ts[:-1], ts[1:])
differences = jax.vmap(lambda ti, tj: tj - ti)(ts[:-1], ts[1:])
print(ts[:6])
print(differences[:6])
print(vals.h[:6])
print(vals.W[:6])
print(vals.H[:6])

[0.4179353  0.52991754 0.5894369  0.6499695  0.82372373 0.9456226 ]
[0.11198223 0.05951935 0.06053263 0.17375422 0.12189889 0.31082076]
[0.11198223 0.05951935 0.06053264 0.1737542  0.12189889 0.31082073]
[ 0.10772365  0.44995847 -0.00810127 -0.40986827 -0.05604121 -0.24828207]
[ 0.04426942  0.11042085 -0.15579225  0.27858496  0.01182858  0.02763878]


In [3]:
ts = jrandom.permutation(permute_key, ts)

# Get some random paths
bm_keys = jrandom.split(bm_key, 1000)
path = jax.vmap(
    lambda k: diffrax.VirtualBrownianTree(t0=t0, t1=t1, shape=(), tol=2**-7, key=k)
)(bm_keys)

# Sample some points
out = []
for ti in ts:
    vals = jax.vmap(lambda p: p.evaluate(t0, ti, use_hh=True))(path)
    out.append((ti, vals))
out = sorted(out, key=lambda x: x[0])

t: 2.9093546867370605, w: [-2.9830499  1.5053439  0.601706  -1.272824  -1.0341576], H: [ 0.4090827   0.4262923   0.06277044 -0.21479373  0.2842798 ], bm: BMInc(h=f32[5], W=f32[5], J=None, H=f32[5])


In [11]:
# Test their conditional statistics
for i in range(1, len(ts) - 2):
    s, bm_s = out[i - 1]
    r, bm_r = out[i]
    u, bm_u = out[i + 1]

    w_s, hh_s = bm_s.wh()
    w_r, hh_r = bm_r.wh()
    w_u, hh_u = bm_u.wh()

    s = s - t0
    r = r - t0
    u = u - t0

    su = u - s
    sr = r - s
    ru = u - r
    d = jnp.sqrt(jnp.power(sr, 3) + jnp.power(ru, 3))
    a = (1 / (2 * su * d)) * jnp.power(sr, 7 / 2) * jnp.sqrt(ru)
    b = (1 / (2 * su * d)) * jnp.power(ru, 7 / 2) * jnp.sqrt(sr)
    c = (1.0 / (jnp.sqrt(12) * d)) * jnp.power(sr, 3 / 2) * jnp.power(ru, 3 / 2)

    hh_su = (1.0 / su) * (u * hh_u - s * hh_s - u / 2 * w_s + s / 2 * w_u)

    w_mean = w_s + (sr / su) * (w_u - w_s) + (6 * sr * ru / jnp.square(su)) * hh_su
    w_std = 2 * (a + b) / su
    normalised_w = (w_r - w_mean) / w_std
    hh_mean = (
        (s / r) * hh_s
        + (jnp.power(sr, 3) / (r * jnp.square(su))) * hh_su
        + 0.5 * w_s
        - s / (2 * r) * w_mean
    )
    hh_var = jnp.square(c / r) + jnp.square(a / r + (s * (a + b)) / (r * su))
    hh_std = jnp.sqrt(hh_var)
    normalised_hh = (hh_r - hh_mean) / hh_std

    h_norm_mean = jnp.mean(normalised_hh)
    h_norm_std = jnp.std(normalised_hh)

    # w_mean_err = jnp.mean(w_mean - w_r)
    # w_var_err = w_std - jnp.std(w_r - w_mean)
    # print(f"mean err: {w_mean_err}, var err: {w_var_err}")

    _, pval_w = stats.kstest(normalised_w, stats.norm.cdf)
    _, pval_hh = stats.kstest(normalised_hh, stats.norm.cdf)

    # Raise if the failure is statistically significant at 10%, subject to
    # multiple-testing correction.
    print(f"pval_W: {pval_w:.8f}, pval_H: {pval_hh:.8f}")
    # print(f"H mean: {h_norm_mean}, H std: {h_norm_std}")

pval_W: 0.38599481, pval_H: 0.82631438
pval_W: 0.15977976, pval_H: 0.09895532
pval_W: 0.67680189, pval_H: 0.39387402
pval_W: 0.10799607, pval_H: 0.06243627
pval_W: 0.28191867, pval_H: 0.20120862
pval_W: 0.12743455, pval_H: 0.18551637
pval_W: 0.92420903, pval_H: 0.88628544
pval_W: 0.72199328, pval_H: 0.84497659
pval_W: 0.17308720, pval_H: 0.23562536
pval_W: 0.59274157, pval_H: 0.49192855
pval_W: 0.89942145, pval_H: 0.93336897
pval_W: 0.52385262, pval_H: 0.29440378
pval_W: 0.17788643, pval_H: 0.21152355
pval_W: 0.73333471, pval_H: 0.66580353
pval_W: 0.49108986, pval_H: 0.45264930
pval_W: 0.54191508, pval_H: 0.53687857
pval_W: 0.05588207, pval_H: 0.02543564
pval_W: 0.20775127, pval_H: 0.19131516
pval_W: 0.40103046, pval_H: 0.39108584
pval_W: 0.69220727, pval_H: 0.65437153
pval_W: 0.30380855, pval_H: 0.28657643
pval_W: 0.67590180, pval_H: 0.73663636
pval_W: 0.98543931, pval_H: 0.95855351
pval_W: 0.04741885, pval_H: 0.06456553
pval_W: 0.76926040, pval_H: 0.83676577
pval_W: 0.37610051, pval_

In [32]:
key = jrandom.PRNGKey(200)
shape_struct = jax.ShapeDtypeStruct((10000,), jnp.float32)
tree1 = VirtualBrownianTree(t0=0, t1=50.3, tol=2**-7, shape=shape_struct, key=key)
tree2 = VirtualBrownianTree(t0=20.3, t1=50.3, tol=2**-7, shape=shape_struct, key=key)

In [23]:
def print_correct_vars(_t0, _t1):
    h = _t1 - _t0
    print(f"true hh_var: {1/12 * h:.4f}, w_var: {h:.4f}")

In [36]:
t0, t1, t2, t3 = 20.55, 20.5533, 20.625, 20.75

In [18]:
w02 = tree2.evaluate(t0, t2)
w13 = tree2.evaluate(t1, t3)
w01 = tree2.evaluate(t0, t1)
w23 = tree2.evaluate(t2, t3)
w03 = tree2.evaluate(t0, t3)

In [37]:
w02, hh02 = tree1.evaluate(t0, t2, use_hh=True).wh()
w13, hh13 = tree1.evaluate(t1, t3, use_hh=True).wh()
w01, hh01 = tree1.evaluate(t0, t1, use_hh=True).wh()
w23, hh23 = tree1.evaluate(t2, t3, use_hh=True).wh()
w03, hh03 = tree1.evaluate(t0, t3, use_hh=True).wh()
wtot, hhtot = tree1.evaluate(0, 1, use_hh=True).wh()

In [38]:
for w, hh in [(w02, hh02), (w13, hh13), (w01, hh01), (w23, hh23), (wtot, hhtot)]:
    print(
        f"hh_mean: {jnp.mean(hh):.6f}, w_mean: {jnp.mean(w):.6f}, "
        f"hh_var: {jnp.var(hh):.5f}, w_var: {jnp.var(w):.5f}"
    )

hh_mean: -0.001077, w_mean: -0.002011, hh_var: 0.00631, w_var: 0.07408
hh_mean: -0.000947, w_mean: 0.004151, hh_var: 0.01642, w_var: 0.18779
hh_mean: 0.000253, w_mean: -0.001249, hh_var: 0.00027, w_var: 0.00328
hh_mean: 0.000603, w_mean: 0.004912, hh_var: 0.01055, w_var: 0.12150
hh_mean: 0.005227, w_mean: -0.027667, hh_var: 0.08260, w_var: 1.00014


In [39]:
print_correct_vars(t0, t2)
print(f"emp  hh_var: {jnp.var(hh02):.4f}, w_var: {jnp.var(w02):.4f}")
print_correct_vars(t1, t3)
print(f"emp  hh_var: {jnp.var(hh13):.4f}, w_var: {jnp.var(w13):.4f}")
print_correct_vars(t0, t1)
print(f"emp  hh_var: {jnp.var(hh01):.4f}, w_var: {jnp.var(w01):.4f}")
print_correct_vars(t2, t3)
print(f"emp  hh_var: {jnp.var(hh23):.4f}, w_var: {jnp.var(w23):.4f}")

true hh_var: 0.0062, w_var: 0.0750
emp  hh_var: 0.0063, w_var: 0.0741
true hh_var: 0.0164, w_var: 0.1967
emp  hh_var: 0.0164, w_var: 0.1878
true hh_var: 0.0003, w_var: 0.0033
emp  hh_var: 0.0003, w_var: 0.0033
true hh_var: 0.0104, w_var: 0.1250
emp  hh_var: 0.0105, w_var: 0.1215


In [41]:
print(jnp.cov(w02, w13))
print(jnp.cov(hh02, hh13))

[[0.45072773 0.16195504]
 [0.16195504 0.3477899 ]]
[[ 0.0362108  -0.01186689]
 [-0.01186689  0.02895591]]


In [23]:
# Chen's relation
w_diff_013 = jnp.mean(jnp.abs(w03 - (w01 + w13)))
hh_chen_01_03 = (
    (t1 - t0) * hh01
    + (t3 - t1) * hh13
    + 0.5 * ((t3 - t0) * w01 - (t1 - t0) * (w01 + w13))
)
hh_diff_013 = jnp.mean(jnp.abs((t3 - t0) * hh03 - hh_chen_01_03))
w_diff_023 = jnp.mean(jnp.abs(w03 - (w02 + w23)))
print(w_diff_013)
print(hh_diff_013)
print(w_diff_023)

1.1658482e-08
1.5718944e-08
1.0407902e-08
